In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
import cv2
import mediapipe as mp
import csv
import os

# 1. ตั้งค่าการใช้งาน MediaPipe Hands
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,        
    max_num_hands=1, # แนะนำให้เริ่มเก็บทีละ 1 มือเพื่อสร้างชุดข้อมูลที่นิ่งและแม่นยำก่อนครับ               
    min_detection_confidence=0.7,   
    min_tracking_confidence=0.7     
)

# --- [ส่วนที่เพิ่มใหม่: ตั้งค่าไฟล์ CSV สำหรับเก็บเฉพาะ 6 จุด] ---
csv_filename = "asl_6points_dataset.csv"
target_ids = [0, 4, 8, 12, 16, 20] # Wrist, Thumb, Index, Middle, Ring, Pinky

if not os.path.exists(csv_filename):
    with open(csv_filename, mode='w', newline='') as f:
        writer = csv.writer(f)
        header = ['label']
        # สร้างหัวตาราง เช่น wrist_x, wrist_y, wrist_z, thumb_x, ...
        point_names = ['wrist', 'thumb', 'index', 'middle', 'ring', 'pinky']
        for name in point_names:
            header.extend([f'{name}_x', f'{name}_y', f'{name}_z'])
        writer.writerow(header)
# -------------------------------------------------------------

hand_address_data = {}
cap = cv2.VideoCapture(0)

print("=== ระบบบันทึกข้อมูลภาษามือ (เฉพาะปลายนิ้ว + ข้อมือ) ===")
print("วิธีใช้: ทำท่าค้างไว้แล้วกดปุ่มตัวอักษรบนคีย์บอร์ด (เช่น 'a', 'b', 'c') เพื่อบันทึกข้อมูล | กด 'ESC' เพื่อปิด")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        print("ไม่สามารถโหลดภาพจากกล้องได้")
        break

    frame = cv2.flip(frame, 1)
    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(image_rgb)

    hand_address_data = {}
    text_y_pos = 30 

    # ตรวจจับปุ่มที่กดบนคีย์บอร์ด
    key = cv2.waitKey(1) & 0xFF
    if key == 27: # 27 คือปุ่ม ESC
        break
    
    # ตรวจสอบว่าเป็นปุ่มตัวอักษรภาษาอังกฤษไหม ถ้าใช่ให้แปลงเป็นตัวพิมพ์ใหญ่
    pressed_char = chr(key).upper() if (97 <= key <= 122 or 65 <= key <= 90) else None

    if results.multi_hand_landmarks and results.multi_handedness:
        for idx, (hand_landmarks, handedness) in enumerate(zip(results.multi_hand_landmarks, results.multi_handedness)):
            hand_label = handedness.classification[0].label 
            
            landmarks_list = []
            for id, lm in enumerate(hand_landmarks.landmark):
                h, w, c = frame.shape
                cx, cy = int(lm.x * w), int(lm.y * h)
                
                landmarks_list.append({
                    'id': id,         
                    'x_pixel': cx,    
                    'y_pixel': cy,    
                    'x_norm': lm.x,   
                    'y_norm': lm.y,   
                    'z_norm': lm.z    
                })
            
            address_key = f"{hand_label}_Hand"
            hand_address_data[address_key] = landmarks_list

            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            
            text_color = (206, 22, 242) if hand_label == "Right" else (255, 255, 0)
            header_text = f"Detected: {address_key}"
            cv2.putText(frame, header_text, (20, text_y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.7, text_color, 2)
            text_y_pos += 25

            finger_tips = {'Thumb': 4, 'Index': 8, 'Middle': 12, 'Ring': 16, 'Pinky': 20}
            for name, tip_id in finger_tips.items():
                pt = landmarks_list[tip_id]
                coord_text = f" - {name} (ID {tip_id}): X={pt['x_pixel']}, Y={pt['y_pixel']}"
                cv2.putText(frame, coord_text, (30, text_y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 10, 0), 1)
                text_y_pos += 20
            
            # --- [ส่วนที่เพิ่มใหม่: ดึงข้อมูลและบันทึกค่า x_norm, y_norm, z_norm ลง CSV] ---
            if pressed_char:
                features_to_save = []
                # วนลูปดึงเฉพาะ ID 0, 4, 8, 12, 16, 20 เท่านั้น
                for target_id in target_ids:
                    pt = landmarks_list[target_id]
                    # เก็บเฉพาะค่า x_norm, y_norm, z_norm ตามที่คุณต้องการ
                    features_to_save.extend([pt['x_norm'], pt['y_norm'], pt['z_norm']])
                
                # บันทึกลงไฟล์ CSV (ตัวอักษรนำหน้าตามด้วยค่าพิกัด 18 ค่า)
                with open(csv_filename, mode='a', newline='') as f:
                    writer = csv.writer(f)
                    writer.writerow([pressed_char] + features_to_save)
                
                # แสดงสถานะบนจอว่ากำลังบันทึกตัวอักษรอะไรอยู่
                cv2.putText(frame, f"SAVING DATA FOR: {pressed_char}", (20, h - 20), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            # ----------------------------------------------------------------------------
            
            text_y_pos += 15 

    cv2.imshow('Hand & Finger Detection', frame)

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

print(f"ปิดกล้องเรียบร้อย! ข้อมูลบันทึกอยู่ในไฟล์ '{csv_filename}'")

=== ระบบบันทึกข้อมูลภาษามือ (เฉพาะปลายนิ้ว + ข้อมือ) ===
วิธีใช้: ทำท่าค้างไว้แล้วกดปุ่มตัวอักษรบนคีย์บอร์ด (เช่น 'a', 'b', 'c') เพื่อบันทึกข้อมูล | กด 'ESC' เพื่อปิด
ปิดกล้องเรียบร้อย! ข้อมูลบันทึกอยู่ในไฟล์ 'asl_6points_dataset.csv'


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pickle

# 1. โหลดข้อมูลจากไฟล์ CSV ที่คุณเพิ่งสร้างขึ้นมา
dataset_path = 'asl_6points_dataset.csv'
data = pd.read_csv(dataset_path)

# ตรวจสอบเบื้องต้นว่ามีข้อมูลในไฟล์ไหม
if len(data) == 0:
    print("❌ ไม่พบข้อมูลในไฟล์ CSV กรุณาไปรันโค้ดเก็บข้อมูล (เปิดกล้องกดปุ่มบันทึก) ให้มีข้อมูลก่อนนะครับ!")
    exit()

print(f" Loaded data successfully! Total samples: {len(data)} rows")

# 2. แยก Features (ค่าพิกัด X, Y, Z ทั้ง 18 ค่า) และ Labels (ตัวอักษรเป้าหมาย)
X = data.drop('label', axis=1).values # เอาทุกคอลัมน์ยกเว้นคอลัมน์ label
y = data['label'].values              # เอาเฉพาะคอลัมน์ label

# 3. แบ่งข้อมูลเป็น 2 ส่วน: สำหรับฝึกสอน 80% และสำหรับทดสอบความแม่นยำ 20%
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(" Training AI Model (Random Forest)...")

# 4. สร้างและฝึกฝนโมเดล Machine Learning
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 5. วัดผลความแม่นยำของโมเดล
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"🎯 Model Accuracy (ความแม่นยำ): {accuracy * 100:.2f}%")

# 6. บันทึกโมเดลที่ฉลาดแล้วเก็บไว้เป็นไฟล์ เพื่อเอาไปใช้เปิดกล้องแปลจริงในพาร์ทถัดไป
model_filename = 'asl_6points_model.p'
with open(model_filename, 'wb') as f:
    pickle.dump({'model': model}, f)

print(f" Saved trained model as '{model_filename}' successfully!")
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pickle

# 1. โหลดข้อมูลจากไฟล์ CSV ของคุณ
dataset_path = 'asl_6points_dataset.csv'
data = pd.read_csv(dataset_path)

print(f" Loaded data: {len(data)} rows")

# 2. แยกตัวแปร Features (พิกัดมือ) และ Label (ตัวอักษร)
X = data.drop('label', axis=1).values 
y = data['label'].values              

# 3. แบ่งข้อมูลเอาไว้สอน 80% และเอาไว้ทดสอบความแม่นยำ 20%
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(" Training AI Model...")

# 4. ใช้สมองกลแบบ Random Forest ในการเรียนรู้
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 5. เช็คความแม่นยำว่า AI ตัวนี้ฉลาดแค่ไหน
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"🎯 ความแม่นยำของ AI ตัวนี้อยู่ที่: {accuracy * 100:.2f}%")

# 6. เซฟสมองกลนี้เก็บไว้เป็นไฟล์ชื่อ 'asl_model.p' เพื่อเอาไปใช้แปลจริง
with open('asl_model.p', 'wb') as f:
    pickle.dump({'model': model}, f)

print(" Saved model successfully!")


 Loaded data successfully! Total samples: 252 rows


ValueError: Input contains NaN